# Telemetry Analysis Pipeline - MVP Testing

This notebook tests the MVP implementation of the telemetry analysis pipeline (Steps 1-5).

**Pipeline Steps:**
1. **Data Loading & Cleaning** - Load and validate telemetry data
2. **Baseline Computation** - Calculate historical percentile baselines
3. **Signal Evaluation** - Score each signal against baselines
4. **Component Aggregation** - Aggregate signals to component level
5. **Machine Aggregation** - Aggregate components to machine level
6. **Output Generation** - Write Golden layer files

**Test Data:**
- Client: `cda`
- Evaluation Week: 50
- Evaluation Year: 2025

In [1]:
# Import required modules
import sys
from pathlib import Path

# Add src to path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
from datetime import datetime

# Import telemetry modules
from src.telemetry import data_loader, data_cleaner, baseline, scoring, aggregation, output_writer
from src.utils.logger import logger

print("✓ All modules imported successfully")

✓ All modules imported successfully


In [2]:
df = pd.read_parquet("../data/telemetry/silver/cda/Telemetry_Wide_With_States")
df.head()

,Fecha,Unit,Estado,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,AirFltr,CnkcasePres,...,LtRBrkTemp,Payload,RAftrclrTemp,RtExhTemp,RtFBrkTemp,RtLtExhTemp,RtRBrkTemp,StrgOilTemp,TCOutTemp,TrnLubeTemp
0,2025-01-04 13:18:00,T_12,Operacional,ND,Cargado,-30.253490,-71.091233,1004.105882,3.686355,-0.039465,...,82.368056,193.3,64.053571,452.338160,34.194444,-4.545761,89.127674,68.00000,88.149666,89.963370
1,2025-01-04 00:05:00,T_12,Operacional,Operacional Bajo,Sin Carga,-30.254291,-71.091413,1001.690000,0.940833,0.155318,...,71.600000,0.0,33.696429,239.846830,29.291667,-4.227229,80.000000,68.37500,82.613889,84.351042
2,2025-01-01 23:48:00,T_12,Operacional,Operacional Alto,Sin Carga,NaN,NaN,1032.469091,1.675595,0.175234,...,83.426840,0.0,43.500000,201.080847,31.284722,-2.361279,87.575758,65.42803,87.381239,86.329023
3,2025-01-03 00:14:00,T_12,Operacional,ND,Cargado,-30.246838,-71.099568,944.506897,2.400871,0.287575,...,68.208333,183.8,34.216667,299.669368,26.625000,NaN,76.375000,68.75000,77.633677,81.538091
4,2025-01-05 13:20:00,T_12,Ralenti,Ralenti Alto,Sin Carga,-30.254721,-71.094740,1009.097826,1.412500,0.183102,...,81.274414,0.0,48.255952,157.766434,30.858974,-3.085166,87.166667,68.00000,88.017857,87.989936


## Configuration

Set pipeline parameters for testing.

In [3]:
# Pipeline configuration
CLIENT = 'cda'
EVALUATION_WEEK = 50
EVALUATION_YEAR = 2025
LOOKBACK_DAYS = 7*16
BASELINE_VERSION = datetime.now().strftime('%Y%m%d')

print(f"Configuration:")
print(f"  Client: {CLIENT}")
print(f"  Evaluation: Week {EVALUATION_WEEK}, Year {EVALUATION_YEAR}")
print(f"  Baseline lookback: {LOOKBACK_DAYS} days")
print(f"  Baseline version: {BASELINE_VERSION}")

Configuration:
  Client: cda
  Evaluation: Week 50, Year 2025
  Baseline lookback: 112 days
  Baseline version: 20260412


---
## Step 1: Data Loading & Cleaning

Load the evaluation week data and apply cleaning pipeline.

In [ ]:
# Step 1A: Load evaluation week
print("=" * 60)
print("STEP 1A: Loading Evaluation Week")
print("=" * 60)

current_df = data_loader.load_evaluation_week(
    client=CLIENT,
    week=EVALUATION_WEEK,
    year=EVALUATION_YEAR
)

print(f"\n✓ Loaded {len(current_df)} rows")
print(f"  Date range: {current_df['Fecha'].min()} to {current_df['Fecha'].max()}")
print(f"  Units: {current_df['Unit'].nunique()}")
print(f"\nFirst few rows:")
current_df.head()

STEP 1A: Loading Evaluation Week
2026-04-12 20:50:09,075 - telemetry - INFO - Loading evaluation week: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\silver\cda\Telemetry_Wide_With_States\Week50Year2025.parquet
2026-04-12 20:50:09,136 - telemetry - INFO - Loaded 81902 rows, 11 units for Week 50 Year 2025

✓ Loaded 81902 rows
  Date range: 2025-12-08 00:00:00 to 2025-12-14 23:59:00
  Units: 11

First few rows:


,Fecha,Unit,Estado,EstadoMaquina,EstadoCarga,GPSLat,GPSLon,GPSElevation,AirFltr,CnkcasePres,...,LtRBrkTemp,Payload,RAftrclrTemp,RtExhTemp,RtFBrkTemp,RtLtExhTemp,RtRBrkTemp,StrgOilTemp,TCOutTemp,TrnLubeTemp
0,2025-12-11 01:47:00,T_12,Ralenti,Ralenti Alto,Cargado,-30.254222,-71.096982,1024.819564,2.083333,0.163897,...,82.160714,173.899994,52.830556,438.468842,86.558333,NaN,89.555263,62.875000,86.328195,89.662549
1,2025-12-09 12:53:00,T_12,Operacional,Operacional Alto,Cargado,-30.243501,-71.093918,927.416669,3.093750,0.272345,...,72.000000,180.500000,54.643939,570.839195,82.458333,-6.383275,80.541667,65.041667,82.558333,85.335870
2,2025-12-10 06:26:00,T_12,Ralenti,Ralenti Alto,Sin Carga,-30.243432,-71.094826,932.413560,0.975920,0.278241,...,73.578782,0.000000,37.389286,205.241936,76.276224,-0.245473,82.329060,60.500000,80.525000,83.630583
3,2025-12-12 03:05:00,T_12,Operacional,Operacional Alto,Cargado,-30.245495,-71.091505,887.989999,2.165972,0.153591,...,67.375000,183.800003,34.260198,460.179545,76.100000,-5.509175,75.250000,58.000000,77.849697,78.997222
4,2025-12-14 01:22:00,T_12,Operacional,Operacional Bajo,Cargado,-30.247535,-71.091714,872.979548,NaN,0.084457,...,71.041667,175.100006,34.250000,187.505321,83.500000,4.230704,80.428571,63.000000,83.510036,84.432500


In [ ]:
# Step 1B: Get signal columns
signal_cols = data_loader.get_signal_columns(current_df)

print(f"✓ Identified {len(signal_cols)} signal columns")
print(f"\nSignal columns: {signal_cols[:10]}...")  # Show first 10

✓ Identified 18 signal columns

Signal columns: ['AirFltr', 'CnkcasePres', 'DiffLubePres', 'DiffTemp', 'EngCoolTemp', 'EngOilFltr', 'EngOilPres', 'LtExhTemp', 'LtFBrkTemp', 'LtRBrkTemp']...


In [ ]:
# Step 1C: Clean data
print("\n" + "=" * 60)
print("STEP 1C: Cleaning Data")
print("=" * 60)

current_df_clean = data_cleaner.clean_telemetry_data(current_df, signal_cols)

print(f"\n✓ Cleaning complete")
print(f"  Final rows: {len(current_df_clean)}")
print(f"  Rows removed: {len(current_df) - len(current_df_clean)}")


STEP 1C: Cleaning Data
2026-04-12 20:50:09,185 - telemetry - INFO - Starting data cleaning pipeline
2026-04-12 20:50:09,265 - telemetry - INFO - Timestamp validation: 81902 valid rows
2026-04-12 20:50:09,329 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(100.0)}
2026-04-12 20:50:09,355 - telemetry - WARNING - Flagged 26 extreme outliers as NaN
2026-04-12 20:50:09,357 - telemetry - INFO - Data cleaning complete: 81902 rows (0.0% removed)

✓ Cleaning complete
  Final rows: 81902
  Rows removed: 0


In [ ]:
# Step 1D: Load component mapping
print("\n" + "=" * 60)
print("STEP 1D: Loading Component Mapping")
print("=" * 60)

component_mapping = data_loader.load_component_mapping(CLIENT)

print(f"\n✓ Loaded mapping for {len(component_mapping)} components")
for comp_name, comp_config in list(component_mapping.items())[:5]:
    print(f"  - {comp_name}: {len(comp_config['signals'])} signals, criticality={comp_config['criticality']}")


STEP 1D: Loading Component Mapping
2026-04-12 20:50:09,375 - telemetry - INFO - Loading component mapping from c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\component_signals_mapping.json
2026-04-12 20:50:09,395 - telemetry - INFO - Loaded mapping for 4 components

✓ Loaded mapping for 4 components
  - Motor: 9 signals, criticality=High
  - Tren de fuerza: 4 signals, criticality=High
  - Frenos: 4 signals, criticality=Medium
  - Direccion: 1 signals, criticality=Medium


---
## Step 2: Baseline Computation

Load historical data and compute percentile baselines.

In [ ]:
# Step 2A: Load baseline training window
print("\n" + "=" * 60)
print("STEP 2A: Loading Baseline Training Window")
print("=" * 60)

baseline_training_df = data_loader.load_baseline_training_window(
    client=CLIENT,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    lookback_days=LOOKBACK_DAYS
)

print(f"\n✓ Loaded {len(baseline_training_df)} historical rows")
print(f"  Date range: {baseline_training_df['Fecha'].min()} to {baseline_training_df['Fecha'].max()}")
print(f"  Units: {baseline_training_df['Unit'].nunique()}")


STEP 2A: Loading Baseline Training Window
2026-04-12 20:50:09,402 - telemetry - INFO - Loading baseline data from 2025-08-25 to 2025-12-14
2026-04-12 20:50:12,715 - telemetry - INFO - Loaded 1247220 rows, 11 units for baseline training

✓ Loaded 1247220 historical rows
  Date range: 2025-08-25 00:00:00 to 2025-12-14 00:00:00
  Units: 11


In [ ]:
# Step 2B: Clean baseline data
print("\n" + "=" * 60)
print("STEP 2B: Cleaning Baseline Data")
print("=" * 60)

baseline_training_clean = data_cleaner.clean_telemetry_data(baseline_training_df, signal_cols)

print(f"\n✓ Baseline data cleaned")
print(f"  Final rows: {len(baseline_training_clean)}")


STEP 2B: Cleaning Baseline Data
2026-04-12 20:50:12,789 - telemetry - INFO - Starting data cleaning pipeline
2026-04-12 20:50:13,453 - telemetry - INFO - Timestamp validation: 1247220 valid rows
2026-04-12 20:50:13,896 - telemetry - WARNING - Removed 9597 duplicate readings
2026-04-12 20:50:14,230 - telemetry - WARNING - Signals with >50% missing data: {'EngOilFltr': np.float64(99.15555867982415)}
2026-04-12 20:50:14,992 - telemetry - WARNING - Flagged 2825 extreme outliers as NaN
2026-04-12 20:50:14,992 - telemetry - INFO - Data cleaning complete: 1237623 rows (0.8% removed)

✓ Baseline data cleaned
  Final rows: 1237623


In [ ]:
# Step 2C: Compute baseline percentiles
print("\n" + "=" * 60)
print("STEP 2C: Computing Baseline Percentiles")
print("=" * 60)

baseline_df = baseline.compute_baseline_percentiles(
    training_df=baseline_training_clean,
    signal_cols=signal_cols,
    baseline_date=BASELINE_VERSION
)

print(f"\n✓ Computed {len(baseline_df)} baseline combinations")
print(f"\nBaseline summary:")
baseline_df.head(10)


STEP 2C: Computing Baseline Percentiles
2026-04-12 20:50:15,010 - telemetry - INFO - Computing baseline percentiles for 18 signals
2026-04-12 20:50:15,011 - telemetry - INFO -   Percentiles: [0.01, 0.05, 0.95, 0.99]
2026-04-12 20:50:15,052 - telemetry - INFO -   Training data: 1237623 rows, 11 units
2026-04-12 20:50:24,985 - telemetry - INFO - Computed 932 baseline combinations
2026-04-12 20:50:24,986 - telemetry - INFO -   Units with baselines: 11
2026-04-12 20:50:24,987 - telemetry - INFO -   Signals with baselines: 18
2026-04-12 20:50:24,988 - telemetry - INFO -   State-specific: 932, Aggregate: 0

✓ Computed 932 baseline combinations

Baseline summary:


,Unit,Signal,EstadoMaquina,P1,P5,P95,P99,sample_count,baseline_version
0,T_10,AirFltr,ND,0.273662,0.583385,4.455224,5.125000,90184,20260412
1,T_10,CnkcasePres,ND,-0.455331,-0.307461,0.253175,0.307115,114592,20260412
2,T_10,DiffLubePres,ND,17.779677,33.600170,114.447415,122.990563,82623,20260412
3,T_10,DiffTemp,ND,33.731907,62.120989,84.063095,86.195762,114843,20260412
4,T_10,EngCoolTemp,ND,53.125000,72.000000,84.875000,87.083333,114627,20260412
5,T_10,EngOilPres,ND,333.445057,363.295760,487.980833,506.491718,113812,20260412
6,T_10,LtExhTemp,ND,111.793974,166.501319,556.557985,578.630005,114859,20260412
7,T_10,LtFBrkTemp,ND,39.045238,70.583333,89.875000,93.607626,114813,20260412
8,T_10,LtRBrkTemp,ND,41.000000,70.583333,90.213992,93.961530,114819,20260412
9,T_10,RAftrclrTemp,ND,19.000000,24.958333,66.125000,71.836250,114710,20260412


In [ ]:
# Step 2D: Save baseline
print("\n" + "=" * 60)
print("STEP 2D: Saving Baseline")
print("=" * 60)

baseline_path = baseline.save_baseline(baseline_df, CLIENT)

print(f"\n✓ Baseline saved to: {baseline_path}")


STEP 2D: Saving Baseline
2026-04-12 20:50:25,049 - telemetry - INFO - Saved baseline to c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_20260412.parquet
2026-04-12 20:50:25,051 - telemetry - INFO -   Baseline version: 20260412
2026-04-12 20:50:25,052 - telemetry - INFO -   Total records: 932

✓ Baseline saved to: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_20260412.parquet


---
## Step 3: Signal Evaluation

Score each signal against baseline percentiles.

In [ ]:
# Step 3: Evaluate signals
print("\n" + "=" * 60)
print("STEP 3: Signal Evaluation")
print("=" * 60)

signal_evaluation_df = scoring.evaluate_signals(
    current_df=current_df_clean,
    baseline_df=baseline_df,
    signal_cols=signal_cols,
    component_mapping=component_mapping
)

print(f"\n✓ Evaluated {len(signal_evaluation_df)} signal-unit combinations")
print(f"\nStatus distribution:")
print(signal_evaluation_df['signal_status'].value_counts(normalize=True))

print(f"\nSample evaluations:")
signal_evaluation_df.head(10)


STEP 3: Signal Evaluation
2026-04-12 20:50:25,059 - telemetry - INFO - Starting signal evaluation
2026-04-12 20:50:25,064 - telemetry - INFO -   Units to evaluate: 11
2026-04-12 20:50:25,065 - telemetry - INFO -   Signals to evaluate: 18


KeyError: 'P2'

In [ ]:
# Analyze signal evaluation results
print("Signal Evaluation Analysis:")
print(f"\nTop 10 signals with highest anomaly rates:")
top_anomalies = signal_evaluation_df.nlargest(10, 'anomaly_percentage')[
    ['unit_id', 'signal_name', 'component', 'signal_status', 'anomaly_percentage', 'window_score_normalized']
]
top_anomalies

Signal Evaluation Analysis:

Top 10 signals with highest anomaly rates:


,unit_id,signal_name,component,signal_status,anomaly_percentage,window_score_normalized
137,T_18,DiffLubePres,Tren de fuerza,Anormal,69.723210,1.916654
141,T_18,EngOilPres,Motor,Anormal,59.577610,1.540688
142,T_18,LtExhTemp,Motor,Anormal,55.265500,1.486206
145,T_18,RtExhTemp,Motor,Anormal,54.420053,1.482794
34,T_12,AirFltr,Motor,Anormal,51.368180,1.411273
21,T_11,EngCoolTemp,Motor,Alerta,50.599718,0.883467
119,T_17,AirFltr,Motor,Anormal,49.616670,1.347356
131,T_17,RtLtExhTemp,Motor,Anormal,48.430214,1.212931
85,T_15,AirFltr,Motor,Anormal,47.000618,1.276850
124,T_17,EngOilPres,Motor,Anormal,45.868726,1.001081


---
## Step 4: Component Aggregation

Aggregate signal scores to component level.

In [ ]:
# Step 4: Aggregate to components
print("\n" + "=" * 60)
print("STEP 4: Component Aggregation")
print("=" * 60)

component_df = aggregation.aggregate_to_components(
    signal_evaluation_df=signal_evaluation_df,
    component_mapping=component_mapping,
    current_df=current_df_clean,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    baseline_version=BASELINE_VERSION
)

print(f"\n✓ Aggregated to {len(component_df)} component-unit combinations")
print(f"\nComponent status distribution:")
print(component_df['component_status'].value_counts())

print(f"\nSample component evaluations:")
component_df.head(10)


STEP 4: Component Aggregation
2026-02-25 15:20:38,233 - telemetry - INFO - Aggregating signals to components
2026-02-25 15:20:38,315 - telemetry - INFO - Component aggregation complete: 44 component-unit combinations
2026-02-25 15:20:38,316 - telemetry - INFO -   Status distribution: {'Normal': 32, 'Alerta': 7, 'Anormal': 5}

✓ Aggregated to 44 component-unit combinations

Component status distribution:
component_status
Normal     32
Alerta      7
Anormal     5
Name: count, dtype: int64

Sample component evaluations:


,unit_id,component,evaluation_week,evaluation_year,component_score,component_status,triggering_signals,signals_evaluation,signal_coverage,sample_count_avg,criticality,baseline_version
0,T_10,Motor,50,2025,0.0375,Normal,[AirFltr],"{'AirFltr': {'status': 'Alerta', 'window_score...",0.888889,6661.125,High,20260225
1,T_10,Tren de fuerza,50,2025,0.0000,Normal,[],"{'DiffLubePres': {'status': 'Normal', 'window_...",1.000000,7143.500,High,20260225
2,T_10,Frenos,50,2025,0.0000,Normal,[],"{'LtFBrkTemp': {'status': 'Normal', 'window_sc...",1.000000,7739.750,Medium,20260225
3,T_10,Direccion,50,2025,0.0000,Normal,[],"{'StrgOilTemp': {'status': 'Normal', 'window_s...",1.000000,7739.000,Medium,20260225
4,T_11,Motor,50,2025,0.4000,Alerta,"[AirFltr, EngCoolTemp, EngOilPres, LtExhTemp, ...","{'AirFltr': {'status': 'Anormal', 'window_scor...",0.888889,8175.625,High,20260225
5,T_11,Tren de fuerza,50,2025,0.0750,Normal,[DiffLubePres],"{'DiffLubePres': {'status': 'Alerta', 'window_...",1.000000,7889.750,High,20260225
6,T_11,Frenos,50,2025,0.0000,Normal,[],"{'LtFBrkTemp': {'status': 'Normal', 'window_sc...",1.000000,8511.500,Medium,20260225
7,T_11,Direccion,50,2025,0.0000,Normal,[],"{'StrgOilTemp': {'status': 'Normal', 'window_s...",1.000000,8511.000,Medium,20260225
8,T_12,Motor,50,2025,0.5250,Anormal,"[AirFltr, CnkcasePres, EngOilPres, LtExhTemp, ...","{'AirFltr': {'status': 'Anormal', 'window_scor...",0.888889,7782.375,High,20260225
9,T_12,Tren de fuerza,50,2025,0.0750,Normal,[DiffLubePres],"{'DiffLubePres': {'status': 'Alerta', 'window_...",1.000000,7514.250,High,20260225


In [ ]:
# Analyze component results
print("Component Analysis:")
print(f"\nTop 10 components with highest scores:")
top_components = component_df.nlargest(10, 'component_score')[
    ['unit_id', 'component', 'component_status', 'component_score', 'triggering_signals']
]
top_components

Component Analysis:

Top 10 components with highest scores:


,unit_id,component,component_status,component_score,triggering_signals
28,T_17,Motor,Anormal,0.6625,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
32,T_18,Motor,Anormal,0.5500,"[EngCoolTemp, EngOilFltr, EngOilPres, LtExhTem..."
16,T_14,Motor,Anormal,0.5375,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
8,T_12,Motor,Anormal,0.5250,"[AirFltr, CnkcasePres, EngOilPres, LtExhTemp, ..."
20,T_15,Motor,Anormal,0.4875,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
40,T_9,Motor,Alerta,0.4500,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."
4,T_11,Motor,Alerta,0.4000,"[AirFltr, EngCoolTemp, EngOilPres, LtExhTemp, ..."
33,T_18,Tren de fuerza,Alerta,0.3250,"[DiffLubePres, TrnLubeTemp]"
34,T_18,Frenos,Alerta,0.3000,"[LtFBrkTemp, LtRBrkTemp, RtFBrkTemp, RtRBrkTemp]"
12,T_13,Motor,Alerta,0.1875,"[AirFltr, EngOilPres, LtExhTemp, RAftrclrTemp,..."


---
## Step 5: Machine Aggregation

Aggregate component evaluations to machine level.

In [ ]:
unit = 'T_18'

unit_components = component_df[component_df['unit_id'] == unit]
        
# Count components by status
status_counts = unit_components['component_status'].value_counts()
components_normal = status_counts.get('Normal', 0)
components_alerta = status_counts.get('Alerta', 0)
components_anormal = status_counts.get('Anormal', 0)

print(f"Unit {unit} component status counts:"
      f"\n  Normal: {components_normal}"
      f"\n  Alerta: {components_alerta}"
      f"\n  Anormal: {components_anormal}")

Unit T_18 component status counts:
  Normal: 1
  Alerta: 2
  Anormal: 1


In [ ]:
# Step 5: Aggregate to machines
print("\n" + "=" * 60)
print("STEP 5: Machine Aggregation")
print("=" * 60)

# Get expected fleet for handling missing units
from pathlib import Path
machine_status_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'machine_status.parquet'
expected_units = aggregation.get_expected_fleet(
    baseline_df=baseline_df,
    previous_machine_status_path=machine_status_path
)

print(f"Expected fleet size: {len(expected_units)} units")
print(f"Units with component data: {component_df['unit_id'].nunique()} units")

machine_df = aggregation.aggregate_to_machines(
    component_df=component_df,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    baseline_version=BASELINE_VERSION,
    expected_units=expected_units,
    component_mapping=component_mapping
)

print(f"\n✓ Aggregated to {len(machine_df)} machines")
print(f"\nMachine status distribution:")
print(machine_df['overall_status'].value_counts())

print(f"\nMachine evaluation results:")
machine_df.head(11)


STEP 5: Machine Aggregation
Expected fleet size: 11 units
Units with component data: 11 units
2026-02-25 15:20:38,387 - telemetry - INFO - Aggregating components to machines
2026-02-25 15:20:38,426 - telemetry - INFO - Machine aggregation complete: 11 machines evaluated
2026-02-25 15:20:38,428 - telemetry - INFO -   Units with data: 11
2026-02-25 15:20:38,429 - telemetry - INFO -   Units without data: 0
2026-02-25 15:20:38,431 - telemetry - INFO -   Overall status distribution: {'Normal': 6, 'Alerta': 4, 'Anormal': 1}
2026-02-25 15:20:38,434 - telemetry - INFO -   Normal: 6
2026-02-25 15:20:38,435 - telemetry - INFO -   Alerta: 4
2026-02-25 15:20:38,436 - telemetry - INFO -   Anormal: 1
2026-02-25 15:20:38,437 - telemetry - INFO -   InsufficientData: 0

✓ Aggregated to 11 machines

Machine status distribution:
overall_status
Normal     6
Alerta     4
Anormal    1
Name: count, dtype: int64

Machine evaluation results:


,unit_id,overall_status,machine_score,priority_score,components_normal,components_alerta,components_anormal,components_insufficient,total_components,evaluation_week,evaluation_year,baseline_version,component_details
0,T_10,Normal,0.00,0.00,4,0,0,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Normal', 's..."
1,T_11,Normal,0.30,10.30,3,1,0,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Alerta', 's..."
2,T_12,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Anormal', '..."
3,T_13,Normal,0.30,10.30,3,1,0,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Alerta', 's..."
4,T_14,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Anormal', '..."
5,T_15,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Anormal', '..."
6,T_16,Normal,0.42,20.42,2,2,0,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Alerta', 's..."
7,T_17,Alerta,1.00,101.00,3,0,1,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Anormal', '..."
8,T_18,Anormal,1.42,121.42,1,2,1,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Anormal', '..."
9,T_24,Normal,0.00,0.00,4,0,0,0,4,50,2025,20260225,"[{'component': 'Motor', 'status': 'Normal', 's..."


In [ ]:
# Analyze machine results
print("Machine Analysis:")
print(f"\nMachines by priority (top 10 highest priority):")
top_priority = machine_df.nlargest(10, 'priority_score')[
    ['unit_id', 'overall_status', 'machine_score', 'priority_score', 
     'components_anormal', 'components_alerta', 'components_normal']
]
top_priority

Machine Analysis:

Machines by priority (top 10 highest priority):


,unit_id,overall_status,machine_score,priority_score,components_anormal,components_alerta,components_normal
8,T_18,Anormal,1.42,121.42,1,2,1
2,T_12,Alerta,1.00,101.00,1,0,3
4,T_14,Alerta,1.00,101.00,1,0,3
5,T_15,Alerta,1.00,101.00,1,0,3
7,T_17,Alerta,1.00,101.00,1,0,3
6,T_16,Normal,0.42,20.42,0,2,2
1,T_11,Normal,0.30,10.30,0,1,3
3,T_13,Normal,0.30,10.30,0,1,3
10,T_9,Normal,0.30,10.30,0,1,3
0,T_10,Normal,0.00,0.00,0,0,4


---
## Step 6: Output Generation

Write Golden layer outputs.

In [ ]:
# Step 6: Write outputs
print("\n" + "=" * 60)
print("STEP 6: Writing Golden Layer Outputs")
print("=" * 60)

output_writer.write_golden_outputs(
    machine_df=machine_df,
    component_df=component_df,
    client=CLIENT
)

print("\n✓ Golden layer outputs written successfully")


STEP 6: Writing Golden Layer Outputs
2026-02-25 15:20:38,505 - telemetry - INFO - Writing Golden layer outputs to c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda
2026-02-25 15:20:38,508 - telemetry - INFO - Creating new machine_status history file
2026-02-25 15:20:38,520 - telemetry - INFO - Wrote machine_status: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\machine_status.parquet
2026-02-25 15:20:38,521 - telemetry - INFO -   Total historical records: 11
2026-02-25 15:20:38,525 - telemetry - INFO - Creating new classified history file
2026-02-25 15:20:38,541 - telemetry - INFO - Wrote classified: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\classified.parquet
2026-02-25 15:20:38,543 - telemetry - INFO -   Total historical records: 44
2026-02-25 15:20:38,543 - telemetry - INFO - Golden layer outputs complete

✓ Golden layer outputs written successfully


In [ ]:
# Write baseline metadata
output_writer.write_baseline_metadata(
    baseline_df=baseline_df,
    client=CLIENT,
    evaluation_week=EVALUATION_WEEK,
    evaluation_year=EVALUATION_YEAR,
    lookback_days=LOOKBACK_DAYS
)

print("✓ Baseline metadata written")

2026-02-25 15:20:38,563 - telemetry - INFO - Wrote baseline metadata: c:\Users\patri\Coddi\Proyectos\telemetry_dashboard\data\telemetry\golden\cda\baselines\baseline_metadata.json
✓ Baseline metadata written


In [ ]:
# Verify time-series behavior
print("\n" + "=" * 60)
print("TIME-SERIES VERIFICATION")
print("=" * 60)

# Load classified to check historical records
classified_path = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT / 'classified.parquet'
if classified_path.exists():
    classified_check = pd.read_parquet(classified_path)
    
    print(f"\nTotal historical records in classified.parquet: {len(classified_check)}")
    
    # Check unique evaluation periods
    periods = classified_check[['evaluation_year', 'evaluation_week']].drop_duplicates().sort_values(['evaluation_year', 'evaluation_week'])
    print(f"\nEvaluation periods in history: {len(periods)}")
    print(periods.to_string(index=False))
    
    # Show sample component evolution for one unit
    if not classified_check.empty:
        sample_unit = classified_check['unit_id'].iloc[0]
        sample_component = classified_check[classified_check['unit_id'] == sample_unit]['component'].iloc[0]
        
        evolution = classified_check[
            (classified_check['unit_id'] == sample_unit) & 
            (classified_check['component'] == sample_component)
        ][['evaluation_week', 'evaluation_year', 'component_status', 'component_score']].sort_values(['evaluation_year', 'evaluation_week'])
        
        print(f"\nComponent evolution example: {sample_unit} - {sample_component}")
        print(evolution.to_string(index=False))
else:
    print("No classified.parquet file found yet")

print("\n" + "=" * 60)
print("NOTE: machine_status.parquet is overwritten (latest only)")
print("      classified.parquet appends history (time-series)")
print("=" * 60)


TIME-SERIES VERIFICATION

Total historical records in classified.parquet: 44

Evaluation periods in history: 1
 evaluation_year  evaluation_week
            2025               50

Component evolution example: T_10 - Motor
 evaluation_week  evaluation_year component_status  component_score
              50             2025           Normal           0.0375

NOTE: machine_status.parquet is overwritten (latest only)
      classified.parquet appends history (time-series)


---
## Validation & Summary

Verify outputs and generate summary statistics.

In [ ]:
# Verify output files exist
from pathlib import Path

base_dir = Path.cwd().parent / 'data' / 'telemetry' / 'golden' / CLIENT
output_files = [
    'machine_status.parquet',
    'classified.parquet'
]

print("Output File Verification:")
for file in output_files:
    file_path = base_dir / file
    if file_path.exists():
        size_mb = file_path.stat().st_size / (1024 * 1024)
        print(f"  ✓ {file} ({size_mb:.2f} MB)")
    else:
        print(f"  ✗ {file} - NOT FOUND")

Output File Verification:
  ✓ machine_status.parquet (0.01 MB)
  ✓ classified.parquet (0.04 MB)


In [ ]:
# Generate pipeline summary
print("\n" + "=" * 60)
print("PIPELINE EXECUTION SUMMARY")
print("=" * 60)

print(f"\nConfiguration:")
print(f"  Client: {CLIENT}")
print(f"  Evaluation: Week {EVALUATION_WEEK}, Year {EVALUATION_YEAR}")
print(f"  Baseline: {BASELINE_VERSION} ({LOOKBACK_DAYS} days lookback)")

print(f"\nData Processing:")
print(f"  Evaluation rows: {len(current_df_clean)}")
print(f"  Baseline rows: {len(baseline_training_clean)}")
print(f"  Units evaluated: {machine_df['unit_id'].nunique()}")

print(f"\nBaseline Computation:")
print(f"  Total baselines: {len(baseline_df)}")
print(f"  State-specific: {(baseline_df['EstadoMaquina'] != 'All').sum()}")
print(f"  Aggregate: {(baseline_df['EstadoMaquina'] == 'All').sum()}")

print(f"\nSignal Evaluation:")
print(f"  Total evaluations: {len(signal_evaluation_df)}")
status_counts = signal_evaluation_df['signal_status'].value_counts()
for status, count in status_counts.items():
    pct = (count / len(signal_evaluation_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\nComponent Aggregation:")
print(f"  Total components: {len(component_df)}")
comp_status_counts = component_df['component_status'].value_counts()
for status, count in comp_status_counts.items():
    pct = (count / len(component_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\nMachine Aggregation:")
print(f"  Total machines: {len(machine_df)}")
machine_status_counts = machine_df['overall_status'].value_counts()
for status, count in machine_status_counts.items():
    pct = (count / len(machine_df)) * 100
    print(f"    {status}: {count} ({pct:.1f}%)")

print(f"\n{'=' * 60}")
print("✓ PIPELINE EXECUTION COMPLETE")
print('=' * 60)


PIPELINE EXECUTION SUMMARY

Configuration:
  Client: cda
  Evaluation: Week 50, Year 2025
  Baseline: 20260225 (112 days lookback)

Data Processing:
  Evaluation rows: 81902
  Baseline rows: 1237623
  Units evaluated: 11

Baseline Computation:
  Total baselines: 932
  State-specific: 932
  Aggregate: 0

Signal Evaluation:
  Total evaluations: 184
    Normal: 119 (64.7%)
    Alerta: 40 (21.7%)
    Anormal: 24 (13.0%)
    InsufficientData: 1 (0.5%)

Component Aggregation:
  Total components: 44
    Normal: 32 (72.7%)
    Alerta: 7 (15.9%)
    Anormal: 5 (11.4%)

Machine Aggregation:
  Total machines: 11
    Normal: 6 (54.5%)
    Alerta: 4 (36.4%)
    Anormal: 1 (9.1%)

✓ PIPELINE EXECUTION COMPLETE


---
## Next Steps

The MVP pipeline is now complete and tested. You can:

1. **Review outputs** in `data/telemetry/golden/{client}/`
2. **Load outputs** for dashboard visualization
3. **Tune thresholds** in scoring and aggregation modules
4. **Extend pipeline** with Phase 2 features (LLM, LSTM, forecasting)

For production use, create a main pipeline script that orchestrates all steps.